# Truth-Check Demo — Reproducing the Companion Article's Claim Map

This notebook reproduces **Plot 3** of the companion article
_"Ein Truth-Check-Protokoll für AI-Forschungs-Output"_
([mybytes.com/research/truth-check-protocol](https://mybytes.com/research/truth-check-protocol)).

It loads the claim map and anchor mapping CSVs, joins them, and
produces a heatmap that visualises which protocol steps have been
completed per atomar claim of the article.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TEMPLATES = REPO_ROOT / 'templates'
FIG_OUT = REPO_ROOT / 'figures'
FIG_OUT.mkdir(exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Templates from: {TEMPLATES}')
print(f'Figures to:     {FIG_OUT}')

In [ ]:
claim_map = pd.read_csv(TEMPLATES / 'claim_map.csv')
anchor_map = pd.read_csv(TEMPLATES / 'anchor_mapping.csv')

joined = claim_map.merge(anchor_map, on='claim_id', how='left')
joined.head()

## Build the Truth-Check status matrix

Convert the seven protocol step statuses into a numeric matrix so we
can render them as a heatmap. Encoding:

- `2` — pass / present
- `1` — partial / pending
- `0` — fail / absent
- `nan` — not applicable

In [ ]:
STATUS_ENCODING = {
    'yes': 2, 'pass': 2, 'high': 2,
    'partial': 1, 'pending': 1, 'medium': 1,
    'no': 0, 'fail': 0, 'low': 0,
    'n/a': np.nan, '': np.nan,
}

TIER_ENCODING = {'T1': 2, 'T2': 2, 'T3': 1, 'T4': 1}

def _encode(value, mapping):
    if isinstance(value, float) and np.isnan(value):
        return np.nan
    key = str(value).strip().lower()
    for k, v in mapping.items():
        if k.lower() == key:
            return v
    return np.nan

matrix_cols = ['anchor_tier', 'bundle_present', 'steel_man_addressed', 'limitations_section', 'reviewer_pass']
matrix_labels = ['Anchor Tier', 'Repro Bundle', 'Steel-Man', 'Limitations', 'Reviewer']

rows = []
for _, r in joined.iterrows():
    rows.append([
        _encode(r['anchor_tier'], TIER_ENCODING),
        _encode(r['bundle_present'], STATUS_ENCODING),
        _encode(r['steel_man_addressed'], STATUS_ENCODING),
        _encode(r['limitations_section'], STATUS_ENCODING),
        _encode(r['reviewer_pass'], STATUS_ENCODING),
    ])

matrix = pd.DataFrame(rows, columns=matrix_labels, index=joined['claim_id'])
matrix

In [ ]:
# Use the executive plot library if available; fall back to inline rendering
try:
    sys.path.insert(0, str(REPO_ROOT.parent.parent.parent.parent / 'src'))
    from common.executive_plots import executive_claim_heatmap
    fig, ax = executive_claim_heatmap(
        matrix,
        claim_texts=joined['claim_text'].tolist(),
        title='Wie dieser Artikel auf seine eigenen Behauptungen geprüft wurde',
        caption='Das Protokoll wird angewendet, nicht behauptet.',
    )
except Exception:
    fig, ax = plt.subplots(figsize=(11, 5))
    cmap = sns.color_palette('RdYlGn', n_colors=3)
    sns.heatmap(matrix, annot=False, cmap=cmap, vmin=0, vmax=2, cbar=False, linewidths=0.6, linecolor='white', ax=ax)
    ax.set_title('Wie dieser Artikel auf seine eigenen Behauptungen geprüft wurde', loc='left', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Claim #')
    fig.text(0.02, -0.02, 'Das Protokoll wird angewendet, nicht behauptet.', fontsize=9, style='italic')

fig.tight_layout()
fig.savefig(FIG_OUT / 'plot3_claim_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

## What this notebook proves

If this notebook runs end-to-end and the figure above renders, then:

1. The article's Plot 3 is reproducible from a fresh environment.
2. The claim map underlying the article is auditable as data, not
   as prose.
3. Any reader can replace `claim_map.csv` and `anchor_mapping.csv`
   with their own and apply the same audit to a different article,
   pitch or PoC report.

That is what "Reproducibility Bundle" means in Step 4 of the
protocol.